# RAG and Agentic Search

**Khóa: Claude with the Anthropic API (Anthropic Skilljar)**

### Vấn đề
Claude **không biết**:
- Dữ liệu riêng của bạn (tài liệu công ty, database nội bộ, ghi chú cá nhân) 📂
- Thông tin sau thời điểm huấn luyện (knowledge cutoff) ⏰

Nếu hỏi về những thứ này, Claude có thể **bịa ra** (hallucination).

### Giải pháp: RAG (Retrieval-Augmented Generation)
**Tìm kiếm (Retrieval)** thông tin liên quan từ kho dữ liệu của bạn → **nhét (Augment)** vào prompt → Claude **trả lời (Generation)** dựa trên đúng thông tin đó.

> 🔑 Ý tưởng cốt lõi: *"Đừng bắt Claude nhớ mọi thứ — hãy đưa đúng tài liệu cho nó đọc trước khi trả lời."*


In [ ]:
!pip install anthropic

In [ ]:
import anthropic, json
client = anthropic.Anthropic()
MODEL = "claude-opus-4-8"

## 1. Pipeline RAG cổ điển (Tổng quan)

```
[Tài liệu]
   │ 1. Chia nhỏ (Chunking)
   ▼
[Các đoạn nhỏ - chunks]
   │ 2. Tạo vector (Embeddings)
   ▼
[Vector Store / Kho vector]
                                  ┌── 3. Câu hỏi -> vector
                                  ▼
                          4. Tìm chunk GIỐNG nhất (Semantic Search)
                                  ▼
                          5. Nhét chunk vào prompt -> Claude trả lời
```

- **Chunking**: chia tài liệu dài thành đoạn nhỏ vừa phải.
- **Embeddings**: biến văn bản thành vector số — văn bản có nghĩa giống nhau thì vector gần nhau.
- **Semantic Search**: tìm theo *ý nghĩa*, không phải khớp từ khóa cứng.


## 2. Kho tài liệu mẫu (Knowledge Base)

Giả sử đây là tài liệu nội bộ về chính sách công ty — thứ Claude chắc chắn không biết.


In [ ]:
documents = [
    "Nhân viên được nghỉ phép năm 12 ngày, cộng thêm 1 ngày cho mỗi 2 năm thâm niên.",
    "Công ty hỗ trợ làm việc từ xa tối đa 3 ngày mỗi tuần, cần đăng ký trước với quản lý.",
    "Chi phí đi lại công tác được hoàn 100% nếu có hóa đơn hợp lệ, nộp trong vòng 30 ngày.",
    "Giờ làm việc tiêu chuẩn là 9h-18h, nghỉ trưa 1 tiếng từ 12h đến 13h.",
    "Nhân viên mới có 2 tháng thử việc, được hưởng 85% lương chính thức.",
    "Công ty đóng bảo hiểm sức khỏe cho nhân viên và 1 người thân phụ thuộc.",
]

## 3. Embeddings — biến văn bản thành vector

Anthropic **không** có API embeddings riêng; tài liệu khuyến nghị dùng **Voyage AI**. Trong production bạn sẽ làm như dưới (cần key Voyage riêng):

```python
import voyageai
vo = voyageai.Client()
vectors = vo.embed(documents, model="voyage-3").embeddings
```

➡️ Để notebook chạy được **không cần key khác**, ở phần dưới ta dùng một hàm tìm kiếm **đơn giản theo từ khóa** để minh họa *cơ chế retrieval*. Nguyên lý (tìm chunk liên quan nhất) là giống nhau.


In [ ]:
import re

def simple_search(query, docs, top_k=2):
    """Tìm kiếm đơn giản: chấm điểm theo số từ trùng giữa câu hỏi và tài liệu.
    (Production: thay bằng semantic search dùng embeddings + vector DB.)"""
    def tokenize(text):
        return set(re.findall(r"\w+", text.lower()))

    q_words = tokenize(query)
    scored = []
    for doc in docs:
        overlap = len(q_words & tokenize(doc))
        scored.append((overlap, doc))

    scored.sort(reverse=True, key=lambda x: x[0])
    return [doc for score, doc in scored[:top_k] if score > 0]

# Thử
print(simple_search("Tôi được nghỉ phép bao nhiêu ngày?", documents))

## 4. RAG cổ điển: Retrieve → Augment → Generate

Ghép retrieval vào prompt. So sánh: hỏi Claude **không có** vs **có** context.


In [ ]:
def ask_without_rag(question):
    res = client.messages.create(
        model=MODEL, max_tokens=512,
        messages=[{"role": "user", "content": question}],
    )
    return res.content[0].text

def ask_with_rag(question):
    # 1. Retrieve
    chunks = simple_search(question, documents)
    context = "\n".join(f"- {c}" for c in chunks)

    # 2. Augment: nhét context vào prompt (dùng thẻ XML cho rõ ràng)
    prompt = f"""
Chỉ dựa vào thông tin trong <tai_lieu> để trả lời. Nếu không có thông tin, hãy nói "Tôi không có thông tin này".

<tai_lieu>
{context}
</tai_lieu>

Câu hỏi: {question}
""".strip()

    # 3. Generate
    res = client.messages.create(
        model=MODEL, max_tokens=512,
        messages=[{"role": "user", "content": prompt}],
    )
    return res.content[0].text

In [ ]:
q = "Tôi làm 4 năm rồi thì được nghỉ phép bao nhiêu ngày một năm?"

print("❌ KHÔNG RAG (Claude đoán/bịa):")
print(ask_without_rag(q))

print("\n✅ CÓ RAG (dựa trên tài liệu thật):")
print(ask_with_rag(q))

## 5. Hạn chế của RAG cổ điển

RAG cổ điển = **tìm 1 lần rồi trả lời**. Nó yếu khi:
- Câu hỏi cần **nhiều bước tìm kiếm** (tìm A → dựa vào A để tìm B).
- Câu hỏi mơ hồ, cần **tinh chỉnh truy vấn**.
- Cần **kết hợp** thông tin từ nhiều lần tìm khác nhau.

➡️ Đây là lúc **Agentic Search** tỏa sáng.


## 6. Agentic Search — để Claude TỰ tìm kiếm 🔍🤖

Thay vì pipeline cứng "tìm 1 lần", ta **biến tìm kiếm thành một công cụ (tool)** và để **Claude tự quyết định**: tìm gì, khi nào tìm, có cần tìm lại không.

Đây chính là **Tool Use áp dụng cho retrieval** — kết hợp bài học trước. Claude trở thành một "agent" chủ động khám phá dữ liệu.


In [ ]:
# Định nghĩa công cụ tìm kiếm cho Claude
search_tool = {
    "name": "search_documents",
    "description": (
        "Tìm kiếm trong kho tài liệu nội bộ của công ty về chính sách nhân sự. "
        "Dùng công cụ này bất cứ khi nào cần thông tin về quy định, chính sách công ty. "
        "Có thể gọi nhiều lần với các truy vấn khác nhau nếu cần."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "query": {"type": "string", "description": "Từ khóa hoặc câu hỏi cần tìm"}
        },
        "required": ["query"],
    },
}

def run_search_tool(tool_input):
    chunks = simple_search(tool_input["query"], documents)
    return "\n".join(chunks) if chunks else "Không tìm thấy tài liệu liên quan."

In [ ]:
def agentic_search(question):
    """Vòng lặp agentic: Claude tự tìm kiếm (có thể nhiều lần) rồi trả lời."""
    messages = [{"role": "user", "content": question}]

    while True:
        response = client.messages.create(
            model=MODEL, max_tokens=1024,
            tools=[search_tool], messages=messages,
        )

        if response.stop_reason == "end_turn":
            return next(b.text for b in response.content if b.type == "text")

        messages.append({"role": "assistant", "content": response.content})
        tool_results = []
        for block in response.content:
            if block.type == "tool_use":
                print(f"  🔍 Claude tự tìm: '{block.input['query']}'")
                result = run_search_tool(block.input)
                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": result,
                })
        messages.append({"role": "user", "content": tool_results})

In [ ]:
# Câu hỏi phức tạp cần tìm nhiều mảng thông tin
print("Câu hỏi: Nếu tôi vừa vào thử việc, tôi được làm từ xa không và lương thế nào?\n")
answer = agentic_search(
    "Nếu tôi vừa vào thử việc, tôi được làm việc từ xa không và mức lương ra sao?"
)
print("\n💬 Trả lời:", answer)

## 7. RAG cổ điển vs Agentic Search

| Tiêu chí | RAG cổ điển | Agentic Search |
|---|---|---|
| Số lần tìm | 1 lần cố định | Claude tự quyết, nhiều lần |
| Tinh chỉnh truy vấn | Không | Có (tự sửa query) |
| Câu hỏi nhiều bước | Yếu | Mạnh |
| Độ phức tạp code | Đơn giản | Cần vòng lặp tool use |
| Chi phí / tốc độ | Rẻ, nhanh | Tốn token hơn, chậm hơn |

👉 **Chọn RAG cổ điển** cho câu hỏi đơn giản, tra cứu 1 mảng. **Chọn Agentic Search** cho câu hỏi phức tạp, nhiều bước, cần suy luận trên dữ liệu.


---
## ✅ Tóm tắt

- **RAG** = cho Claude đọc đúng tài liệu trước khi trả lời → tránh bịa, dùng được dữ liệu riêng.
- Pipeline: **Chunk → Embed → Retrieve → Augment → Generate**.
- **Embeddings** = tìm theo *ý nghĩa* (dùng Voyage AI trong production; Anthropic không có embeddings API riêng).
- **Agentic Search** = biến tìm kiếm thành *tool*, để Claude tự tìm nhiều lần & tinh chỉnh → mạnh hơn cho câu hỏi phức tạp.
- Agentic Search = **Tool Use + Retrieval** kết hợp.

> 💡 Trong thực tế, kho vector lớn nên dùng vector database (Pinecone, Chroma, pgvector...) + embeddings thật. Hàm `simple_search` ở đây chỉ để minh họa cơ chế.
